# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_raw_encoding.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_raw_encoding.parquet')

In [4]:
X_train.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,0.226287,0.560855,0.224078,0.163037,0.260953,0.228153
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,0.212810,0.206102,0.224078,0.173927,0.260953,0.222945
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,0.226287,0.560855,0.306270,0.342228,0.260953,0.223490
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,0.226287,0.560855,0.224078,0.342228,0.225075,0.228153
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,0.226287,0.224644,0.224078,0.163037,0.228745,0.223490


In [5]:
X_test.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,
690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,0.226287,0.560855,0.306270,0.342228,0.225075,0.223490
690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,0.236020,0.560855,0.306270,0.163037,0.260953,0.222945
690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,0.236020,0.009132,0.306270,0.342228,0.187756,0.227764
690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,0.226287,0.206102,0.142713,0.173927,0.260953,0.222945
690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,0.226287,0.560855,0.224078,0.342228,0.225075,0.222945


In [6]:
cat_features_raw = X_train.select_dtypes(include=['category', 'object']).columns.tolist()

X_train = X_train.astype({feat: 'category' for feat in cat_features_raw})
X_test = X_test.astype({feat: 'category' for feat in cat_features_raw})

cat_features_raw

[]

# Machine Learning

In [7]:
models = dict(
    lgbm=load_pickle('../models/layer_1/lightgbm_n_trial_60_data_type_raw_target_encoding.pkl'),
    xgb=load_pickle('../models/layer_1/xgboost_n_trial_60_data_type_raw_target_encoding.pkl'),
    # cat=load_pickle('../models/layer_1/catboost_n_trial_60_data_type_raw.pkl'),
)

## Train Dataset

In [8]:
X_train_stacking = pd.DataFrame({})

In [9]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(
        model, 
        X_train, 
        y_train.health_condition,
        method='predict_proba', 
        cv=StratifiedKFold(shuffle=True, random_state=42, n_splits=3),
        params={'cat_features': cat_features_raw} if model_name == 'cat' else None,
        n_jobs=1 if model_name == 'cat' else 1
    )
    
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|                                                                                                                                                                                        | 0/2 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 50%|███████████████████████████████████████████████████████████████████████████████████████▌                                                                                       | 1/2 [02:57<02:57, 177.68s/it]

Predicting Train Dataset xgb


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [05:09<00:00, 154.70s/it]


## Test Dataset

In [10]:
X_test_stacking = pd.DataFrame({})

In [11]:
for model_name, model in models.items():
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

Predicting Test Dataset lgbm
Predicting Test Dataset xgb


# Saving

In [12]:
X_train_stacking.to_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials_data_type_raw_taret_encoding.parquet')
X_test_stacking.to_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials_data_type_raw_target_encoding.parquet')

In [13]:
X_train_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.003278,0.000834,0.995889,0.005375,0.001396,0.993230
1,0.997061,0.000223,0.002716,0.995312,0.000386,0.004302
2,0.003584,0.000622,0.995794,0.003329,0.000530,0.996142
3,0.003934,0.003009,0.993058,0.004901,0.001941,0.993158
4,0.995190,0.000011,0.004800,0.993787,0.000018,0.006195


In [14]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012261,0.002968,0.984771,0.011874,0.003576,0.984551
1,0.405696,0.002057,0.592248,0.459639,0.002737,0.537624
2,0.999251,0.000597,0.000152,0.998100,0.001247,0.000653
3,0.981066,0.000008,0.018926,0.987928,0.000017,0.012055
4,0.012981,0.002376,0.984643,0.009526,0.002050,0.988424


In [15]:
X_train.shape

(690088, 13)

In [16]:
X_test.shape

(295753, 13)

In [17]:
y_train.head()

,health_condition
id,
0,2
1,0
2,2
3,2
4,0
